In [1]:
#Loading nessary libraries
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import tensorflow as tf
import matplotlib.pyplot as plt

In [2]:
RAW_DIR = Path("../data/raw/asl_alphabet")
MODELS_DIR = Path("../models")
ART_DIR = Path("../artifacts")
PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports/figures")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_SIZE_GRAY = (64, 64)   # for baseline + model2
IMAGE_SIZE_TL   = (96, 96)   # for MobileNetV2
BATCH_SIZE = 32


print("RAW_DIR:", RAW_DIR.exists())
print("MODELS_DIR:", MODELS_DIR.exists())
print("ART_DIR:", ART_DIR.exists())
print("PROCESSED_DIR:", PROCESSED_DIR.exists())

RAW_DIR: True
MODELS_DIR: True
ART_DIR: True
PROCESSED_DIR: True


In [3]:
#Load class mapping
with open(ART_DIR / "class_mapping.json", "r") as f:
    mapping = json.load(f)

classes = mapping["classes"] 
num_classes = len(classes)

print(" num_classes:", num_classes)
print("Sample classes:", classes[:12])

 num_classes: 36
Sample classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b']


In [4]:
#Load test split
test_df = pd.read_csv(PROCESSED_DIR / "test_split.csv")
print("test size:", len(test_df))
test_df.head()


test size: 21340


,path,label,label_id
0,..\data\raw\asl_alphabet\b\B1004.jpg,b,11
1,..\data\raw\asl_alphabet\f\268.jpg,f,15
2,..\data\raw\asl_alphabet\a\A1286.jpg,a,10
3,..\data\raw\asl_alphabet\o\1410.jpg,o,24
4,..\data\raw\asl_alphabet\y\Y335 copy 3.jpg,y,34


In [5]:
#Create tf.data datasets for both models
IMAGE_SIZE_GRAY = (64, 64)   # baseline + model2
IMAGE_SIZE_TL   = (96, 96)   # mobilenetv2

def load_gray64(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, IMAGE_SIZE_GRAY)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

def load_rgb96(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, IMAGE_SIZE_TL)
    img = tf.cast(img, tf.float32) / 255.0
    img = tf.image.grayscale_to_rgb(img)  # convert to (H,W,3)
    return img, label

def make_ds(df, loader_fn):
    ds = tf.data.Dataset.from_tensor_slices((df["path"].values, df["label_id"].values))
    ds = ds.map(loader_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

test_ds_gray64 = make_ds(test_df, load_gray64)
test_ds_rgb96  = make_ds(test_df, load_rgb96)

# y_true is the same labels for both datasets
y_true = np.concatenate([y.numpy() for _, y in test_ds_gray64], axis=0)

print("y_true shape:", y_true.shape)

# sanity check shapes
for x, y in test_ds_gray64.take(1):
    print("gray64 batch:", x.shape, y.shape)
for x, y in test_ds_rgb96.take(1):
    print("rgb96 batch:", x.shape, y.shape)


y_true shape: (21340,)
gray64 batch: (32, 64, 64, 1) (32,)
rgb96 batch: (32, 96, 96, 3) (32,)


In [22]:
model_paths = {
    "baseline": MODELS_DIR / "baseline_cnn.keras",
    "model2_augmented": MODELS_DIR / "model2_augmented.keras",
    "model3_mobilenetv2": MODELS_DIR / "model3_mobilenetv2.keras",}

models = {}
for name, path in model_paths.items():
    if path.exists():
        models[name] = tf.keras.models.load_model(path)
        print("Loaded:", name, "->", path.name, "| input:", models[name].input_shape)
    else:
        print("Missing:", name, "->", path.name)


Loaded: baseline -> baseline_cnn.keras | input: (None, 64, 64, 1)
Loaded: model2_augmented -> model2_augmented.keras | input: (None, 64, 64, 1)
Loaded: model3_mobilenetv2 -> model3_mobilenetv2.keras | input: (None, 96, 96, 3)


In [23]:
def predict_labels(model, ds):
    probs = model.predict(ds, verbose=0)
    y_pred = np.argmax(probs, axis=1)
    y_conf = np.max(probs, axis=1)
    return y_pred, y_conf

In [24]:
def confusion_matrix_np(y_true, y_pred, n):
    cm = np.zeros((n, n), dtype=np.int64)
    for t, p in zip(y_true, y_pred):
        cm[int(t), int(p)] += 1
    return cm

In [25]:
def per_class_metrics(cm, classes):
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0) - tp
    fn = cm.sum(axis=1) - tp

    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp+fp)!=0)
    recall    = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp+fn)!=0)
    f1        = np.divide(2*precision*recall, precision+recall, out=np.zeros_like(tp), where=(precision+recall)!=0)
    support   = cm.sum(axis=1)

    return pd.DataFrame({
        "class": classes,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "support": support})

In [26]:
def top_confusions(cm, classes, k=15):
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                pairs.append((cm[i, j], i, j))
    pairs.sort(reverse=True, key=lambda x: x[0])

    rows = []
    for count, i, j in pairs[:k]:
        rows.append({"count": int(count), "true": classes[i], "pred": classes[j]})
    return pd.DataFrame(rows)

In [27]:
results = []
all_metrics = {}
all_confusions = {}
all_top_confusions = {}

for name, model in models.items():
    # choose correct dataset based on expected input
    input_shape = model.input_shape  # (None, H, W, C)
    h, w, c = input_shape[1], input_shape[2], input_shape[3]

    if (h, w, c) == (64, 64, 1):
        ds = test_ds_gray64
    elif (h, w, c) == (96, 96, 3):
        ds = test_ds_rgb96
    else:
        raise ValueError(f"Unknown input shape for {name}: {input_shape}")

    y_pred, y_conf = predict_labels(model, ds)

    cm = confusion_matrix_np(y_true, y_pred, num_classes)
    metrics_df = per_class_metrics(cm, classes)

    overall_acc = float((y_pred == y_true).mean())
    macro_f1 = float(metrics_df["f1"].mean())
    avg_conf = float(np.mean(y_conf))

    results.append({
        "model": name,
        "test_accuracy": overall_acc,
        "macro_f1": macro_f1,
        "avg_confidence": avg_conf,
        "input_shape": str(input_shape)
    })

    all_confusions[name] = cm
    all_metrics[name] = metrics_df.sort_values("f1")  # hardest at top
    all_top_confusions[name] = top_confusions(cm, classes, k=25)

summary_df = pd.DataFrame(results).sort_values(["macro_f1", "test_accuracy"], ascending=False)
summary_df


,model,test_accuracy,macro_f1,avg_confidence,input_shape
2,model3_mobilenetv2,0.998782,0.998496,0.998756,"(None, 96, 96, 3)"
0,baseline,0.995876,0.995928,0.997202,"(None, 64, 64, 1)"
1,model2_augmented,0.990628,0.989477,0.986507,"(None, 64, 64, 1)"


In [28]:
for name in models.keys():
    print("\n==============================")
    print("MODEL:", name)
    print("Hardest 10 classes (lowest F1):")
    display(all_metrics[name].head(10)[["class","f1","precision","recall","support"]])

    print("\nTop confusion pairs (teaching focus):")
    display(all_top_confusions[name].head(15))



MODEL: baseline
Hardest 10 classes (lowest F1):


,class,f1,precision,recall,support
16,g,0.983051,0.979303,0.986828,911
17,h,0.986180,0.993318,0.979144,911
0,0,0.990121,1.000000,0.980435,460
32,w,0.990446,0.987302,0.993610,313
7,7,0.991361,0.987097,0.995662,461
30,u,0.991770,0.993814,0.989733,487
8,8,0.992333,1.000000,0.984783,460
34,y,0.993179,0.997260,0.989130,368
6,6,0.993464,0.995633,0.991304,460
14,e,0.994583,0.991361,0.997826,460



Top confusion pairs (teaching focus):


,count,true,pred
0,19,h,g
1,7,0,o
2,6,8,7
3,6,g,h
4,4,6,w
5,4,a,e
6,4,u,r
7,4,y,l
8,3,4,5
9,3,c,d



MODEL: model2_augmented
Hardest 10 classes (lowest F1):


...

,count,true,pred
0,47,a,e
1,19,r,u
2,14,w,6
3,13,v,k
4,10,a,b
5,9,2,k
6,8,b,e
7,5,h,g
8,5,w,v
9,4,0,o



MODEL: model3_mobilenetv2
Hardest 10 classes (lowest F1):


,class,f1,precision,recall,support
2,2,0.983264,0.967078,1.000000,235
31,v,0.994881,1.000000,0.989813,589
35,z,0.995502,0.991045,1.000000,332
3,3,0.995763,0.991561,1.000000,235
16,g,0.996143,1.000000,0.992316,911
17,h,0.996173,0.992375,1.000000,911
13,d,0.997800,1.000000,0.995609,911
0,0,0.997821,1.000000,0.995652,460
4,4,0.997831,0.995671,1.000000,460
28,s,0.998597,0.997199,1.000000,356



Top confusion pairs (teaching focus):


,count,true,pred
0,7,g,h
1,6,v,2
2,2,0,o
3,2,d,z
4,2,l,3
5,1,5,4
6,1,6,2
7,1,7,4
8,1,8,z
9,1,d,2


In [29]:
EVAL_DIR = ART_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(EVAL_DIR / "model_comparison_summary.csv", index=False)

for name in models.keys():
    np.save(EVAL_DIR / f"confusion_matrix_{name}.npy", all_confusions[name])
    all_metrics[name].to_csv(EVAL_DIR / f"per_class_metrics_{name}.csv", index=False)
    all_top_confusions[name].to_csv(EVAL_DIR / f"top_confusions_{name}.csv", index=False)

print("Saved evaluation artifacts to:", EVAL_DIR)


Saved evaluation artifacts to: ..\artifacts\evaluation


In [36]:
#Locking the deployment model
DEPLOY_MODEL = "model3_mobilenetv2"
print("Deployment model selected:", DEPLOY_MODEL)


Deployment model selected: model3_mobilenetv2


In [35]:
#Identify the 10 hardest letters for the deployment model
hard_letters = (all_metrics[DEPLOY_MODEL].sort_values("f1").head(10).reset_index(drop=True))

hard_letters


,class,precision,recall,f1,support
0,2,0.967078,1.000000,0.983264,235
1,v,1.000000,0.989813,0.994881,589
2,z,0.991045,1.000000,0.995502,332
3,3,0.991561,1.000000,0.995763,235
4,g,1.000000,0.992316,0.996143,911
5,h,0.992375,1.000000,0.996173,911
6,d,1.000000,0.995609,0.997800,911
7,0,1.000000,0.995652,0.997821,460
8,4,0.995671,1.000000,0.997831,460
9,s,0.997199,1.000000,0.998597,356


In [34]:
#Identify the 10 easiest letters for the deployment model
easy_letters = (all_metrics[DEPLOY_MODEL].sort_values("f1", ascending=False).head(10).reset_index(drop=True))

easy_letters


,class,precision,recall,f1,support
0,u,1.0,1.0,1.0,487
1,w,1.0,1.0,1.0,313
2,f,1.0,1.0,1.0,911
3,e,1.0,1.0,1.0,460
4,x,1.0,1.0,1.0,349
5,y,1.0,1.0,1.0,368
6,m,1.0,1.0,1.0,438
7,n,1.0,1.0,1.0,596
8,i,1.0,1.0,1.0,1021
9,p,1.0,1.0,1.0,385


In [37]:
confusion_pairs = all_top_confusions[DEPLOY_MODEL]
confusion_pairs.head(15)

,count,true,pred
0,7,g,h
1,6,v,2
2,2,0,o
3,2,d,z
4,2,l,3
5,1,5,4
6,1,6,2
7,1,7,4
8,1,8,z
9,1,d,2


In [41]:
#
curriculum = {
    "easy_start": easy_letters["class"].tolist(),
    "hard_focus": hard_letters["class"].tolist(),
    "confusion_pairs": confusion_pairs[["true", "pred"]].head(10).to_dict(orient="records")}

curriculum

{'easy_start': ['u', 'w', 'f', 'e', 'x', 'y', 'm', 'n', 'i', 'p'],
 'hard_focus': ['2', 'v', 'z', '3', 'g', 'h', 'd', '0', '4', 's'],
 'confusion_pairs': [{'true': 'g', 'pred': 'h'},
  {'true': 'v', 'pred': '2'},
  {'true': '0', 'pred': 'o'},
  {'true': 'd', 'pred': 'z'},
  {'true': 'l', 'pred': '3'},
  {'true': '5', 'pred': '4'},
  {'true': '6', 'pred': '2'},
  {'true': '7', 'pred': '4'},
  {'true': '8', 'pred': 'z'},
  {'true': 'd', 'pred': '2'}]}

In [43]:
#Saving curriculum artifacts
CURRICULUM_DIR = ART_DIR / "curriculum"
CURRICULUM_DIR.mkdir(parents=True, exist_ok=True)

hard_letters.to_csv(CURRICULUM_DIR / "hard_letters.csv", index=False)
easy_letters.to_csv(CURRICULUM_DIR / "easy_letters.csv", index=False)
confusion_pairs.to_csv(CURRICULUM_DIR / "confusion_pairs.csv", index=False)

import json
with open(CURRICULUM_DIR / "curriculum.json", "w") as f:
    json.dump(curriculum, f, indent=2)

print("Curriculum artifacts saved to:", CURRICULUM_DIR)


Curriculum artifacts saved to: ..\artifacts\curriculum
